In [0]:
%pip install -U azure-ai-formrecognizer transformers torch pillow pypdfium2
dbutils.library.restartPython()

import os
import io
import numpy as np
import pypdfium2 as pdfium
from PIL import Image

import torch
from transformers import LayoutLMv3Processor, LayoutLMv3Model

from azure.core.credentials import AzureKeyCredential
from azure.ai.formrecognizer import DocumentAnalysisClient

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, ArrayType, FloatType
)

DI_ENDPOINT = "endpoints"
DI_KEY = "key"

SOURCE_TABLE = "noc_documents"
N_DOCS = 3400

RANDOM_SAMPLE = True
RANDOM_SEED = 42

PDF_RENDER_SCALE = 2.0

MAX_PAGES_PER_PDF = 5

MODEL_NAME = "microsoft/layoutlmv3-base"
MAX_TOKENS = 512

WORD_CAP = 400

OUT_PAGES_TABLE = "noc_layoutlmv3_page_embeddings_sample"
OUT_DOCS_TABLE  = "noc_layoutlmv3_doc_embeddings_sample"

PATH_PREFIX_FILTER = None  

pages_schema = StructType([
    StructField("path", StringType(), False),
    StructField("page_num", IntegerType(), False),
    StructField("page_id", StringType(), False),
    StructField("embedding", ArrayType(FloatType()), False),
    StructField("word_count", IntegerType(), False),
])

docs_schema = StructType([
    StructField("path", StringType(), False),
    StructField("doc_id", StringType(), False),
    StructField("embedding", ArrayType(FloatType()), False),
    StructField("page_count", IntegerType(), False),
])

di_client = DocumentAnalysisClient(
    endpoint=DI_ENDPOINT,
    credential=AzureKeyCredential(DI_KEY)
)

processor = LayoutLMv3Processor.from_pretrained(MODEL_NAME, apply_ocr=False)
model = LayoutLMv3Model.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def l2_normalize(v):
    n = float(np.linalg.norm(v))
    if n == 0.0:
        return v.astype(np.float32)
    return (v / n).astype(np.float32)

def normalize_box(box, width, height):
    x0, y0, x1, y1 = box
    x0 = int(1000 * (x0 / width))
    x1 = int(1000 * (x1 / width))
    y0 = int(1000 * (y0 / height))
    y1 = int(1000 * (y1 / height))
    if x0 < 0: x0 = 0
    if y0 < 0: y0 = 0
    if x1 > 1000: x1 = 1000
    if y1 > 1000: y1 = 1000
    return [x0, y0, x1, y1]

def polygon_to_bbox(poly):
    xs = []
    ys = []
    for p in poly:
        if hasattr(p, "x") and hasattr(p, "y"):
            xs.append(float(p.x))
            ys.append(float(p.y))
        else:
            break

    if len(xs) > 0:
        return [min(xs), min(ys), max(xs), max(ys)]

    vals = list(poly)
    xs = [float(vals[i]) for i in range(0, len(vals), 2)]
    ys = [float(vals[i]) for i in range(1, len(vals), 2)]
    return [min(xs), min(ys), max(xs), max(ys)]

def render_pdf_to_pil_images(pdf_local_path, scale=2.0, max_pages=None):
    pdf = pdfium.PdfDocument(pdf_local_path)
    total = len(pdf)
    take = total if max_pages is None else min(total, max_pages)
    images = []
    for i in range(take):
        page = pdf.get_page(i)
        img = page.render(scale=scale).to_pil().convert("RGB")
        images.append(img)
    return images

def di_ocr_words_boxes_from_image_bytes(image_bytes):
    poller = di_client.begin_analyze_document(
        model_id="prebuilt-layout",
        document=image_bytes
    )
    result = poller.result()

    if result.pages is None or len(result.pages) == 0:
        return [], []

    page = result.pages[0]
    words = []
    boxes_pixels = []

    if hasattr(page, "words") and page.words:
        for w in page.words:
            if not w.content or not w.polygon:
                continue
            bbox = polygon_to_bbox(w.polygon)
            words.append(w.content)
            boxes_pixels.append(bbox)

    return words, boxes_pixels

@torch.no_grad()
def embed_page(pil_img, words, boxes_pixels):
    w, h = pil_img.size

    if words is None:
        words = []
    if boxes_pixels is None:
        boxes_pixels = []

    if len(words) == 0 or len(boxes_pixels) == 0:
        return np.zeros((model.config.hidden_size,), dtype=np.float32)

    m = min(len(words), len(boxes_pixels))
    words = words[:m]
    boxes_pixels = boxes_pixels[:m]

    boxes = []
    for b in boxes_pixels:
        boxes.append(normalize_box(b, w, h))

    enc = processor(
        images=pil_img,
        text=words,
        boxes=boxes,
        truncation=True,
        padding="max_length",
        max_length=MAX_TOKENS,
        return_tensors="pt"
    )

    for k in enc:
        enc[k] = enc[k].to(device)

    out = model(**enc)
    cls_vec = out.last_hidden_state[:, 0, :] 
    return cls_vec[0].detach().cpu().numpy().astype(np.float32)

def dbfs_to_local(dbfs_path: str) -> str:
    if dbfs_path.startswith("/dbfs/"):
        return dbfs_path
    if dbfs_path.startswith("dbfs:/"):
        return "/dbfs/" + dbfs_path.replace("dbfs:/", "")
    if dbfs_path.startswith("/FileStore/"):
        return "/dbfs" + dbfs_path
    return dbfs_path  

base = spark.table(SOURCE_TABLE).select("path")

pdf_candidates = base.where(F.lower(F.col("path")).contains(".pdf"))

if PATH_PREFIX_FILTER is not None:
    pdf_candidates = pdf_candidates.where(F.col("path").startswith(PATH_PREFIX_FILTER))

cand_count = pdf_candidates.count()
print("Candidate rows that look like PDFs:", cand_count)

if cand_count == 0:
    print("No PDF-like paths found. Showing 20 sample paths from table:")
    display(base.limit(20))
    empty_pages = spark.createDataFrame([], schema=pages_schema)
    empty_docs  = spark.createDataFrame([], schema=docs_schema)
    empty_pages.write.mode("overwrite").saveAsTable(OUT_PAGES_TABLE)
    empty_docs.write.mode("overwrite").saveAsTable(OUT_DOCS_TABLE)
    raise ValueError("No PDF-like paths matched. Fix your path filter / data first.")

if RANDOM_SAMPLE:
    pdf_candidates = pdf_candidates.orderBy(F.rand(RANDOM_SEED))

sample_df = pdf_candidates.limit(N_DOCS)
paths = [r["path"] for r in sample_df.collect()]

print("PDFs selected:", len(paths))
print("First 3 sample paths:")
for p in paths[:3]:
    print(" -", p)

page_rows = []
doc_rows = []
failed = []

for i, pdf_path in enumerate(paths, start=1):
    pdf_local = dbfs_to_local(pdf_path)
    pdf_name = os.path.basename(pdf_path)

    print(f"[{i}/{len(paths)}] Processing: {pdf_path}")

    try:
        page_images = render_pdf_to_pil_images(
            pdf_local_path=pdf_local,
            scale=PDF_RENDER_SCALE,
            max_pages=MAX_PAGES_PER_PDF
        )

        page_vecs = []

        for page_num, pil_img in enumerate(page_images, start=1):
            buf = io.BytesIO()
            pil_img.save(buf, format="PNG")
            img_bytes = buf.getvalue()

            words, boxes_pixels = di_ocr_words_boxes_from_image_bytes(img_bytes)

            if len(words) > WORD_CAP:
                words = words[:WORD_CAP]
                boxes_pixels = boxes_pixels[:WORD_CAP]

            vec = embed_page(pil_img, words, boxes_pixels)
            vec = l2_normalize(vec)
            page_vecs.append(vec)

            page_id = f"{pdf_name}::page_{page_num:04d}"

            page_rows.append((
                pdf_path,
                int(page_num),
                page_id,
                [float(x) for x in vec.tolist()],
                int(len(words))
            ))

        if len(page_vecs) > 0:
            doc_vec = np.mean(np.stack(page_vecs, axis=0), axis=0).astype(np.float32)
            doc_vec = l2_normalize(doc_vec)
        else:
            doc_vec = np.zeros((model.config.hidden_size,), dtype=np.float32)

        doc_rows.append((
            pdf_path,
            pdf_name,
            [float(x) for x in doc_vec.tolist()],
            int(len(page_vecs))
        ))

    except Exception as e:
        msg = str(e)
        print("FAILED:", pdf_path, "->", msg)
        failed.append((pdf_path, msg))

print("Done.")
print(" - Page rows:", len(page_rows))
print(" - Doc rows :", len(doc_rows))
print(" - Failed   :", len(failed))
if len(failed) > 0:
    print("First failure:", failed[0])

pages_df = spark.createDataFrame(page_rows, schema=pages_schema)
docs_df  = spark.createDataFrame(doc_rows, schema=docs_schema)

pages_df.write.mode("overwrite").saveAsTable(OUT_PAGES_TABLE)
docs_df.write.mode("overwrite").saveAsTable(OUT_DOCS_TABLE)

print("Saved tables:")
print(" -", OUT_PAGES_TABLE, "| rows:", pages_df.count())
print(" -", OUT_DOCS_TABLE,  "| rows:", docs_df.count())

display(docs_df.limit(20))

In [0]:
%sql
DESCRIBE noc_documents;